# 01 - Data Preparation

## Operational Energy Efficiency & Risk Intelligence

### Objective

This notebook prepares the raw industrial energy data for exploratory analysis, time-series analysis, and machine learning.

The dataset contains 15-minute electrical measurements from multiple assets in a manufacturing facility. Most observations are from 2025, with a small number from December 31, 2024.

The main preparation steps are:

- load and identify the relevant raw measurement files
- retain total asset-level electrical measurements
- check missing values, data reliability, and duplicates
- add asset metadata
- validate timestamps and demand values
- create a clean dataset for the next stages of the project

In [1]:
import pandas as pd
import zipfile

## 1. Loading the Raw Data

The measurements are stored in multiple CSV files inside one ZIP file.

In [2]:
zip_path = "../data_raw/Industrial Asset-Level Electrical Energy Dataset from a Manufacturing Facility (15-Minute Aggregates).zip"

## 2. Identifying the Measurement Files

I first check the files inside the ZIP and select the CSV files that contain the asset measurements.

In [3]:
with zipfile.ZipFile(zip_path, "r") as zip_file:
    file_names = zip_file.namelist()

csv_files = [
    name for name in file_names
    if "/data_csv/" in name and name.endswith(".csv")
]

print("Total files:", len(file_names))
print("Measurement CSV files:", len(csv_files))

Total files: 43896
Measurement CSV files: 10943


In [4]:
csv_files[:10]

['Industrial Asset-Level Electrical Energy Dataset from a Manufacturing Facility (15-Minute Aggregates)/data_csv/asset_id=ahu_a/dt_utc=2024-12-31/part-2024-12-31.csv',
 'Industrial Asset-Level Electrical Energy Dataset from a Manufacturing Facility (15-Minute Aggregates)/data_csv/asset_id=ahu_a/dt_utc=2025-01-01/part-2025-01-01.csv',
 'Industrial Asset-Level Electrical Energy Dataset from a Manufacturing Facility (15-Minute Aggregates)/data_csv/asset_id=ahu_a/dt_utc=2025-01-02/part-2025-01-02.csv',
 'Industrial Asset-Level Electrical Energy Dataset from a Manufacturing Facility (15-Minute Aggregates)/data_csv/asset_id=ahu_a/dt_utc=2025-01-03/part-2025-01-03.csv',
 'Industrial Asset-Level Electrical Energy Dataset from a Manufacturing Facility (15-Minute Aggregates)/data_csv/asset_id=ahu_a/dt_utc=2025-01-04/part-2025-01-04.csv',
 'Industrial Asset-Level Electrical Energy Dataset from a Manufacturing Facility (15-Minute Aggregates)/data_csv/asset_id=ahu_a/dt_utc=2025-01-05/part-2025-01-0

## 3. Inspecting One Measurement File

Before combining all files, I inspect one CSV file to understand its structure, columns and data types.

In [5]:
sample_file = csv_files[0]

with zipfile.ZipFile(zip_path, "r") as zip_file:
    with zip_file.open(sample_file) as file:
        sample_df = pd.read_csv(file)

sample_df.head()

,AssetId,Phase,window_start_utc,DateKeyUtc,TimeKeyUtc,WindowStartLocal,HourLocal,DateLocal,Energy_kWh_15m,Demand_kW,AvgPower_kW_15m,Minutes,SecondsObserved,DataCoveragePct,IsReliableWindow,SecondsReliable,ReliableCoveragePct
0,ahu_a,L1,2024-12-31 08:30:00,20241231,83000,2024-12-31 08:30:00,8,2024-12-31,0.000000,0.000000,NaN,13.38,803.0,89.222222,0,0.0,0.000000
1,ahu_a,L2,2024-12-31 08:30:00,20241231,83000,2024-12-31 08:30:00,8,2024-12-31,0.000000,0.000000,NaN,13.38,803.0,89.222222,0,0.0,0.000000
2,ahu_a,L3,2024-12-31 08:30:00,20241231,83000,2024-12-31 08:30:00,8,2024-12-31,0.003267,0.013069,0.014648,13.38,803.0,89.222222,1,803.0,89.222222
3,ahu_a,ALL,2024-12-31 08:30:00,20241231,83000,2024-12-31 08:30:00,8,2024-12-31,0.003267,0.013069,0.014648,15.00,900.0,100.000000,1,803.0,89.222222
4,ahu_a,L1,2024-12-31 08:45:00,20241231,84500,2024-12-31 08:45:00,8,2024-12-31,0.000000,0.000000,NaN,15.00,900.0,100.000000,0,0.0,0.000000


### Structure of the Data

I check the available columns and their data types before deciding which measurements to use.

In [6]:
sample_df.columns.tolist()

['AssetId',
 'Phase',
 'window_start_utc',
 'DateKeyUtc',
 'TimeKeyUtc',
 'WindowStartLocal',
 'HourLocal',
 'DateLocal',
 'Energy_kWh_15m',
 'Demand_kW',
 'AvgPower_kW_15m',
 'Minutes',
 'SecondsObserved',
 'DataCoveragePct',
 'IsReliableWindow',
 'SecondsReliable',
 'ReliableCoveragePct']

In [7]:
sample_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 248 entries, 0 to 247
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   AssetId              248 non-null    str    
 1   Phase                248 non-null    str    
 2   window_start_utc     248 non-null    str    
 3   DateKeyUtc           248 non-null    int64  
 4   TimeKeyUtc           248 non-null    int64  
 5   WindowStartLocal     248 non-null    str    
 6   HourLocal            248 non-null    int64  
 7   DateLocal            248 non-null    str    
 8   Energy_kWh_15m       248 non-null    float64
 9   Demand_kW            248 non-null    float64
 10  AvgPower_kW_15m      129 non-null    float64
 11  Minutes              248 non-null    float64
 12  SecondsObserved      248 non-null    float64
 13  DataCoveragePct      248 non-null    float64
 14  IsReliableWindow     248 non-null    int64  
 15  SecondsReliable      248 non-null    float64
 16  R

### Selecting Total Asset Measurements

The data contains measurements for the three electrical phases (`L1`, `L2`, `L3`) and one combined measurement (`ALL`). For this project, I use `ALL` because it represents the total electrical behavior of each asset.

In [8]:
sample_all = sample_df[sample_df["Phase"] == "ALL"]

sample_all.head()

,AssetId,Phase,window_start_utc,DateKeyUtc,TimeKeyUtc,WindowStartLocal,HourLocal,DateLocal,Energy_kWh_15m,Demand_kW,AvgPower_kW_15m,Minutes,SecondsObserved,DataCoveragePct,IsReliableWindow,SecondsReliable,ReliableCoveragePct
3,ahu_a,ALL,2024-12-31 08:30:00,20241231,83000,2024-12-31 08:30:00,8,2024-12-31,0.003267,0.013069,0.014648,15.0,900.0,100.0,1,803.0,89.222222
7,ahu_a,ALL,2024-12-31 08:45:00,20241231,84500,2024-12-31 08:45:00,8,2024-12-31,0.003604,0.014416,0.014416,15.0,900.0,100.0,1,900.0,100.000000
11,ahu_a,ALL,2024-12-31 09:00:00,20241231,90000,2024-12-31 09:00:00,9,2024-12-31,0.003489,0.013955,0.013955,15.0,900.0,100.0,1,900.0,100.000000
15,ahu_a,ALL,2024-12-31 09:15:00,20241231,91500,2024-12-31 09:15:00,9,2024-12-31,0.003487,0.013948,0.013948,15.0,900.0,100.0,1,900.0,100.000000
19,ahu_a,ALL,2024-12-31 09:30:00,20241231,93000,2024-12-31 09:30:00,9,2024-12-31,0.003733,0.014930,0.014930,15.0,900.0,100.0,1,900.0,100.000000


### Missing Values and Reliability

Before combining all files, I check whether the total measurements contain missing values and how the reliability indicator is distributed.

In [9]:
sample_all.isnull().sum()

AssetId                0
Phase                  0
window_start_utc       0
DateKeyUtc             0
TimeKeyUtc             0
WindowStartLocal       0
HourLocal              0
DateLocal              0
Energy_kWh_15m         0
Demand_kW              0
AvgPower_kW_15m        0
Minutes                0
SecondsObserved        0
DataCoveragePct        0
IsReliableWindow       0
SecondsReliable        0
ReliableCoveragePct    0
dtype: int64

In [10]:
sample_all["IsReliableWindow"].value_counts()

IsReliableWindow
1    62
Name: count, dtype: int64

## 4. Combining the Measurement Files

I now combine all measurement CSV files into one dataframe.

For each file, I keep only rows where `Phase == "ALL"`, because these rows represent the total electrical measurement for the asset rather than the individual phases (`L1`, `L2`, `L3`).

This creates one consistent asset-level dataset for the remaining analysis.

In [11]:
all_data = []

with zipfile.ZipFile(zip_path, "r") as zip_file:

    for file_name in csv_files:

        with zip_file.open(file_name) as file:
            temp_df = pd.read_csv(file)

        temp_df = temp_df[temp_df["Phase"] == "ALL"]

        all_data.append(temp_df)

df = pd.concat(all_data, ignore_index=True)

print(df.shape)

(1039873, 17)


## 5. Checking the Combined Dataset

After combining the files, I check the size, structure and data types of the full dataset.

In [12]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 1039873
Columns: 17


,AssetId,Phase,window_start_utc,DateKeyUtc,TimeKeyUtc,WindowStartLocal,HourLocal,DateLocal,Energy_kWh_15m,Demand_kW,AvgPower_kW_15m,Minutes,SecondsObserved,DataCoveragePct,IsReliableWindow,SecondsReliable,ReliableCoveragePct
0,ahu_a,ALL,2024-12-31 08:30:00,20241231,83000,2024-12-31 08:30:00,8,2024-12-31,0.003267,0.013069,0.014648,15.0,900.0,100.0,1,803.0,89.222222
1,ahu_a,ALL,2024-12-31 08:45:00,20241231,84500,2024-12-31 08:45:00,8,2024-12-31,0.003604,0.014416,0.014416,15.0,900.0,100.0,1,900.0,100.000000
2,ahu_a,ALL,2024-12-31 09:00:00,20241231,90000,2024-12-31 09:00:00,9,2024-12-31,0.003489,0.013955,0.013955,15.0,900.0,100.0,1,900.0,100.000000
3,ahu_a,ALL,2024-12-31 09:15:00,20241231,91500,2024-12-31 09:15:00,9,2024-12-31,0.003487,0.013948,0.013948,15.0,900.0,100.0,1,900.0,100.000000
4,ahu_a,ALL,2024-12-31 09:30:00,20241231,93000,2024-12-31 09:30:00,9,2024-12-31,0.003733,0.014930,0.014930,15.0,900.0,100.0,1,900.0,100.000000


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1039873 entries, 0 to 1039872
Data columns (total 17 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   AssetId              1039873 non-null  str    
 1   Phase                1039873 non-null  str    
 2   window_start_utc     1039873 non-null  str    
 3   DateKeyUtc           1039873 non-null  int64  
 4   TimeKeyUtc           1039873 non-null  int64  
 5   WindowStartLocal     1039873 non-null  str    
 6   HourLocal            1039873 non-null  int64  
 7   DateLocal            1039873 non-null  str    
 8   Energy_kWh_15m       1039873 non-null  float64
 9   Demand_kW            1039873 non-null  float64
 10  AvgPower_kW_15m      1020637 non-null  float64
 11  Minutes              1039873 non-null  float64
 12  SecondsObserved      1039873 non-null  float64
 13  DataCoveragePct      1039873 non-null  float64
 14  IsReliableWindow     1039873 non-null  int64  
 15  SecondsRe

### Missing Values

I check the missing values in the full dataset and calculate the percentage of missing observations in `AvgPower_kW_15m`.

In [14]:
df.isnull().sum()

AssetId                    0
Phase                      0
window_start_utc           0
DateKeyUtc                 0
TimeKeyUtc                 0
WindowStartLocal           0
HourLocal                  0
DateLocal                  0
Energy_kWh_15m             0
Demand_kW                  0
AvgPower_kW_15m        19236
Minutes                    0
SecondsObserved            0
DataCoveragePct            0
IsReliableWindow           0
SecondsReliable            0
ReliableCoveragePct        0
dtype: int64

In [15]:
missing_percentage = df["AvgPower_kW_15m"].isnull().mean() * 100

print("Missing percentage:", round(missing_percentage, 2), "%")

Missing percentage: 1.85 %


### Reliability Check

I check whether the missing `AvgPower_kW_15m` values are connected to measurements marked as unreliable.

In [16]:
df[df["AvgPower_kW_15m"].isnull()]["IsReliableWindow"].value_counts()

IsReliableWindow
0    19236
Name: count, dtype: int64

In [17]:
print(df["IsReliableWindow"].value_counts())

print("\nPercentage:")
print(df["IsReliableWindow"].value_counts(normalize=True) * 100)

IsReliableWindow
1    1016182
0      23691
Name: count, dtype: int64

Percentage:
IsReliableWindow
1    97.721741
0     2.278259
Name: proportion, dtype: float64


### Keeping Reliable Measurements

The missing `AvgPower_kW_15m` values occur only in measurement windows marked as unreliable. Since approximately 97.7% of the observations are reliable, I keep only reliable measurement windows rather than imputing values from unreliable data.

This preserves the original measured values while removing observations that do not meet the dataset's reliability criterion.

In [18]:
df_clean = df[df["IsReliableWindow"] == 1].copy()

print("Rows before cleaning:", len(df))
print("Rows after cleaning:", len(df_clean))

Rows before cleaning: 1039873
Rows after cleaning: 1016182


I check the cleaned data again to confirm that no missing values remain.

In [19]:
df_clean.isnull().sum()

AssetId                0
Phase                  0
window_start_utc       0
DateKeyUtc             0
TimeKeyUtc             0
WindowStartLocal       0
HourLocal              0
DateLocal              0
Energy_kWh_15m         0
Demand_kW              0
AvgPower_kW_15m        0
Minutes                0
SecondsObserved        0
DataCoveragePct        0
IsReliableWindow       0
SecondsReliable        0
ReliableCoveragePct    0
dtype: int64

### Duplicate Check

I check for exact duplicate rows and also for repeated measurements for the same asset at the same timestamp.

In [20]:
print("Exact duplicate rows:", df_clean.duplicated().sum())

print(
    "Duplicate asset-timestamps:",
    df_clean.duplicated(
        subset=["AssetId", "window_start_utc"]
    ).sum()
)

Exact duplicate rows: 0
Duplicate asset-timestamps: 0


### Date and Time

I convert the timestamp to datetime format and check the period covered by the dataset.

In [21]:
df_clean["window_start_utc"] = pd.to_datetime(
    df_clean["window_start_utc"]
)

print("Start date:", df_clean["window_start_utc"].min())
print("End date:", df_clean["window_start_utc"].max())

Start date: 2024-12-31 07:30:00
End date: 2025-12-31 23:45:00


### Assets

I check how many assets are included and which asset IDs are available.

In [22]:
print("Number of assets:", df_clean["AssetId"].nunique())

df_clean["AssetId"].unique()

Number of assets: 43


<ArrowStringArray>
[ 'ahu_a',  'ahu_b',  'ahu_c', 'comp_a', 'comp_b',   'ex_a',   'ex_b',
   'hp_a',   'hp_b',   'mh_a',   'mi_a',   'mi_b',  'mix_a',  'mix_b',
  'mix_c',  'mix_d',    'p_a',    'p_b',    'p_c',    'p_d',    'p_e',
    'p_f',    'p_g',    'p_h',    'p_i',    'p_j',    'p_k',    'p_l',
    'p_m',    'p_n',  'php_c',  'php_j',   'pr_e',   'pr_f',   'pr_k',
   'sb_a',   'sb_b',   'sb_c',    'u_a',    'u_b',    'u_c',    'u_d',
    'u_e']
Length: 43, dtype: str

## 6. Adding Asset Information

The ZIP file also contains metadata about the monitored equipment. I use this file to connect each `AssetId` with its asset type.

In [23]:
other_csv_files = [
    name for name in file_names
    if name.endswith(".csv") and "/data_csv/" not in name
]

other_csv_files

['Industrial Asset-Level Electrical Energy Dataset from a Manufacturing Facility (15-Minute Aggregates)/metadata/AssetList.csv',
 'Industrial Asset-Level Electrical Energy Dataset from a Manufacturing Facility (15-Minute Aggregates)/metadata/EdgeList.csv',
 'Industrial Asset-Level Electrical Energy Dataset from a Manufacturing Facility (15-Minute Aggregates)/metadata/meter_data_quality_log.csv',
 'Industrial Asset-Level Electrical Energy Dataset from a Manufacturing Facility (15-Minute Aggregates)/metadata/schema_gold_table.csv',
 'Industrial Asset-Level Electrical Energy Dataset from a Manufacturing Facility (15-Minute Aggregates)/validation/quantitative_dataset_summary/asset_type_summary.csv',
 'Industrial Asset-Level Electrical Energy Dataset from a Manufacturing Facility (15-Minute Aggregates)/validation/quantitative_dataset_summary/monthly_summary.csv',
 'Industrial Asset-Level Electrical Energy Dataset from a Manufacturing Facility (15-Minute Aggregates)/validation/quantitative_d

In [24]:
asset_file = [
    name for name in other_csv_files
    if name.endswith("/metadata/AssetList.csv")
][0]

with zipfile.ZipFile(zip_path, "r") as zip_file:
    with zip_file.open(asset_file) as file:
        asset_info = pd.read_csv(file)

asset_info.head()

,asset_id,asset_type,em_manufacturer,em_model,em_communication_protocol,em_acquisition_pathway,stream_start_utc,stream_end_utc,is_submeter
0,ahu_a,HVAC_AHU,Weidmüller,EM220,modbus_tcp_ip,direct_modbus_tcp,01/01/2025 00:00,31/12/2025 23:59,True
1,ahu_b,HVAC_AHU,Weidmüller,EM220,modbus_tcp_ip,direct_modbus_tcp,01/01/2025 00:00,31/12/2025 23:59,True
2,ahu_c,HVAC_AHU,Weidmüller,EM220,modbus_tcp_ip,direct_modbus_tcp,01/01/2025 00:00,31/12/2025 23:59,True
3,u_a,Utilities,Weidmüller,EM220,modbus_tcp_ip,direct_modbus_tcp,22/05/2025 13:25,31/12/2025 23:59,True
4,u_b,Utilities,Weidmüller,EM220,modbus_tcp_ip,direct_modbus_tcp,22/05/2025 13:25,31/12/2025 23:59,True


I keep only `asset_id` and `asset_type`, because these are the metadata fields needed for the later analysis and modeling.

`asset_id` identifies the individual equipment, while `asset_type` allows results to be compared across equipment categories.

In [25]:
asset_info[["asset_id", "asset_type"]]

,asset_id,asset_type
0,ahu_a,HVAC_AHU
1,ahu_b,HVAC_AHU
2,ahu_c,HVAC_AHU
3,u_a,Utilities
4,u_b,Utilities
5,u_c,Utilities
6,u_d,Utilities
7,u_e,Utilities
8,comp_a,CompressedAir
9,comp_b,CompressedAir


### Merging the Asset Types

I add the asset type to the cleaned measurements so the later analysis can compare different equipment categories.

In [26]:
rows_before_merge = len(df_clean)

asset_types = asset_info[["asset_id", "asset_type"]].rename(
    columns={"asset_id": "AssetId"}
)

df_clean = df_clean.merge(
    asset_types,
    on="AssetId",
    how="left"
)

df_clean.head()

,AssetId,Phase,window_start_utc,DateKeyUtc,TimeKeyUtc,WindowStartLocal,HourLocal,DateLocal,Energy_kWh_15m,Demand_kW,AvgPower_kW_15m,Minutes,SecondsObserved,DataCoveragePct,IsReliableWindow,SecondsReliable,ReliableCoveragePct,asset_type
0,ahu_a,ALL,2024-12-31 08:30:00,20241231,83000,2024-12-31 08:30:00,8,2024-12-31,0.003267,0.013069,0.014648,15.0,900.0,100.0,1,803.0,89.222222,HVAC_AHU
1,ahu_a,ALL,2024-12-31 08:45:00,20241231,84500,2024-12-31 08:45:00,8,2024-12-31,0.003604,0.014416,0.014416,15.0,900.0,100.0,1,900.0,100.000000,HVAC_AHU
2,ahu_a,ALL,2024-12-31 09:00:00,20241231,90000,2024-12-31 09:00:00,9,2024-12-31,0.003489,0.013955,0.013955,15.0,900.0,100.0,1,900.0,100.000000,HVAC_AHU
3,ahu_a,ALL,2024-12-31 09:15:00,20241231,91500,2024-12-31 09:15:00,9,2024-12-31,0.003487,0.013948,0.013948,15.0,900.0,100.0,1,900.0,100.000000,HVAC_AHU
4,ahu_a,ALL,2024-12-31 09:30:00,20241231,93000,2024-12-31 09:30:00,9,2024-12-31,0.003733,0.014930,0.014930,15.0,900.0,100.0,1,900.0,100.000000,HVAC_AHU


In [27]:
print("Rows before merge:", rows_before_merge)
print("Rows after merge:", len(df_clean))

Rows before merge: 1016182
Rows after merge: 1016182


I check that every asset was successfully matched with an asset type.

In [28]:
df_clean["asset_type"].isnull().sum()

np.int64(0)

### Removing Constant Columns

After filtering the data, I check whether any columns now contain only one value. Columns with no variation do not add useful information to the later analysis.

In [29]:
unique_counts = df_clean.nunique()

constant_columns = unique_counts[unique_counts == 1]

constant_columns

Phase               1
IsReliableWindow    1
dtype: int64

In [30]:
df_clean = df_clean.drop(
    columns=["Phase", "IsReliableWindow"]
)

## 7. Final Data Checks

Before saving the dataset, I confirm that the expected assets and time period are still present after cleaning.

In [31]:
print("Number of assets:", df_clean["AssetId"].nunique())
print("Start date:", df_clean["window_start_utc"].min())
print("End date:", df_clean["window_start_utc"].max())

Number of assets: 43
Start date: 2024-12-31 07:30:00
End date: 2025-12-31 23:45:00


### Sorting the Data

I sort the measurements by asset and time so each asset is in chronological order.

In [32]:
df_clean = df_clean.sort_values(
    ["AssetId", "window_start_utc"]
).reset_index(drop=True)

df_clean.head()

,AssetId,window_start_utc,DateKeyUtc,TimeKeyUtc,WindowStartLocal,HourLocal,DateLocal,Energy_kWh_15m,Demand_kW,AvgPower_kW_15m,Minutes,SecondsObserved,DataCoveragePct,SecondsReliable,ReliableCoveragePct,asset_type
0,ahu_a,2024-12-31 08:30:00,20241231,83000,2024-12-31 08:30:00,8,2024-12-31,0.003267,0.013069,0.014648,15.0,900.0,100.0,803.0,89.222222,HVAC_AHU
1,ahu_a,2024-12-31 08:45:00,20241231,84500,2024-12-31 08:45:00,8,2024-12-31,0.003604,0.014416,0.014416,15.0,900.0,100.0,900.0,100.000000,HVAC_AHU
2,ahu_a,2024-12-31 09:00:00,20241231,90000,2024-12-31 09:00:00,9,2024-12-31,0.003489,0.013955,0.013955,15.0,900.0,100.0,900.0,100.000000,HVAC_AHU
3,ahu_a,2024-12-31 09:15:00,20241231,91500,2024-12-31 09:15:00,9,2024-12-31,0.003487,0.013948,0.013948,15.0,900.0,100.0,900.0,100.000000,HVAC_AHU
4,ahu_a,2024-12-31 09:30:00,20241231,93000,2024-12-31 09:30:00,9,2024-12-31,0.003733,0.014930,0.014930,15.0,900.0,100.0,900.0,100.000000,HVAC_AHU


### Demand Check

Since `Demand_kW` is one of the main variables used later in the project, I check its distribution and confirm that it does not contain negative values.

In [33]:
df_clean["Demand_kW"].describe()

count    1.016182e+06
mean     1.164370e+01
std      3.859929e+01
min      0.000000e+00
25%      2.546649e-02
50%      8.880085e-01
75%      8.038168e+00
max      5.913142e+02
Name: Demand_kW, dtype: float64

In [34]:
(df_clean["Demand_kW"] < 0).sum()

np.int64(0)

### Final Validation

Before saving the cleaned dataset, I run a final validation to confirm the number of rows and assets and to ensure that no missing values or duplicate asset-timestamp combinations remain.

In [35]:
print("Rows:", len(df_clean))
print("Assets:", df_clean["AssetId"].nunique())
print("Missing values:", df_clean.isnull().sum().sum())

print(
    "Duplicate asset-timestamps:",
    df_clean.duplicated(
        subset=["AssetId", "window_start_utc"]
    ).sum()
)

Rows: 1016182


Assets: 43
Missing values: 0
Duplicate asset-timestamps: 0


## 8. Saving the Clean Dataset

The cleaned dataset is saved as one CSV file so it can be used directly in the next notebooks without repeating the preparation steps.

In [36]:
output_path = "../data_processed/manufacturing_energy_clean.csv"

df_clean.to_csv(output_path, index=False)

print("Clean dataset saved.")

Clean dataset saved.


## Summary

The raw measurement files were combined into a single asset-level dataset and filtered to retain total (`ALL`) electrical measurements from reliable measurement windows.

The preparation process included:

- checking missing values and measurement reliability
- removing unreliable observations rather than imputing them
- checking duplicate asset-timestamp combinations
- converting and validating timestamps
- adding asset-type metadata
- removing constant columns
- sorting measurements chronologically by asset
- validating the final demand data

The final cleaned dataset contains **1,016,182 15-minute observations from 43 industrial assets**, with no missing values or duplicate asset-timestamp combinations.

The cleaned data is saved to `data_processed/manufacturing_energy_clean.csv` and is ready for exploratory data analysis and time-series analysis.